**Implementasi Code Tubes DKA Sistem Penilaian Indeks Kualitas Udara (AQI) Berbasis Parameter Polutan**

In [1]:
import pandas as pd  # Impor pustaka pandas untuk membaca dan mengolah data tabel
import numpy as np  # Impor numpy untuk operasi matematika (tidak banyak dipakai langsung tapi berguna)
from sklearn.metrics import mean_absolute_error, mean_squared_error  # Impor fungsi pengukur kesalahan prediksi

# 1. Load & Preprocessing Data
df = pd.read_csv('city_day.csv')  # Baca file CSV berisi data kualitas udara kota per hari
df_clean = df[['PM2.5', 'PM10', 'NO2', 'CO', 'SO2', 'AQI']].dropna()  # Pilih 5 kolom polutan + AQI, buang baris yang ada nilainya kosong
data_sample = df_clean.sample(n=5000, random_state=42).reset_index(drop=True)  # Ambil 5000 data secara acak (agar proses Mamdani tidak terlalu lama), random_state=42 agar hasilnya sama setiap dijalankan

# 2. FUNGSI KEANGGOTAAN (From Scratch)
def trapmf(x, a, b, c, d):  # Fungsi bentuk trapesium: mengukur seberapa besar nilai x termasuk dalam suatu kategori
    if x <= a or x >= d: return 0.0  # Kalau x di luar batas kiri (a) atau kanan (d), berarti tidak masuk kategori sama sekali (nilai 0)
    elif a < x <= b: return (x - a) / (b - a)  # Sisi kiri naik: semakin mendekati b, semakin besar nilainya
    elif b < x <= c: return 1.0  # Di antara b dan c, nilai keanggotaan penuh (= 1, artinya benar-benar masuk kategori ini)
    elif c < x < d: return (d - x) / (d - c)  # Sisi kanan turun: semakin mendekati d, semakin kecil nilainya
    return 0.0  # Jaga-jaga jika tidak ada kondisi yang cocok, kembalikan 0

def trimf(x, a, b, c):  # Fungsi bentuk segitiga: sebenarnya trapesium tapi dengan puncak setitik (b=c)
    return trapmf(x, a, b, b, c)  # Gunakan trapmf tapi titik tengah kiri dan kanan sama (b), membentuk segitiga

# 3. PROSES FUZZIFIKASI
def fuzzify_input(val, param):  # Fungsi untuk mengubah nilai angka menjadi derajat keanggotaan (Rendah/Sedang/Tinggi)
    if param == 'PM2.5':  # Jika parameter yang diproses adalah polutan PM2.5
        return {'Rendah': trapmf(val, 0, 0, 30, 60), 'Sedang': trimf(val, 30, 60, 90), 'Tinggi': trapmf(val, 60, 90, 500, 500)}  # Kembalikan seberapa 'Rendah', 'Sedang', 'Tinggi' nilai PM2.5 tersebut
    elif param == 'PM10':  # Jika parameter yang diproses adalah polutan PM10
        return {'Rendah': trapmf(val, 0, 0, 50, 100), 'Sedang': trimf(val, 50, 100, 150), 'Tinggi': trapmf(val, 100, 150, 500, 500)}  # Batas kategori PM10 berbeda dari PM2.5 karena standar polutan berbeda
    elif param == 'NO2':  # Jika parameter yang diproses adalah polutan NO2
        return {'Rendah': trapmf(val, 0, 0, 40, 80), 'Sedang': trimf(val, 40, 80, 120), 'Tinggi': trapmf(val, 80, 120, 300, 300)}  # Batas kategori untuk gas NO2
    elif param == 'CO':  # Jika parameter yang diproses adalah polutan CO (karbon monoksida)
        return {'Rendah': trapmf(val, 0, 0, 1, 2), 'Sedang': trimf(val, 1, 2, 3), 'Tinggi': trapmf(val, 2, 3, 50, 50)}  # CO diukur dalam satuan kecil (mg/m³), makanya batasnya jauh lebih kecil
    elif param == 'SO2':  # Jika parameter yang diproses adalah polutan SO2
        return {'Rendah': trapmf(val, 0, 0, 40, 80), 'Sedang': trimf(val, 40, 80, 120), 'Tinggi': trapmf(val, 80, 120, 300, 300)}  # Batas kategori SO2 sama dengan NO2

# 4. INFERENSI & RULE BASE
def apply_rules(f_pm25, f_pm10, f_no2, f_co, f_so2):  # Fungsi untuk menerapkan 15 aturan fuzzy berdasarkan nilai keanggotaan semua polutan
    rules = []  # Buat daftar kosong untuk menampung hasil tiap aturan
    
    # R1
    rules.append(('Baik', min(f_pm25['Rendah'], f_pm10['Rendah'], f_no2['Rendah'], f_co['Rendah'], f_so2['Rendah'])))  # Aturan 1: Jika SEMUA polutan Rendah → udara Baik (ambil nilai terkecil karena operator AND)
    # R2
    rules.append(('Buruk', min(f_pm25['Tinggi'], f_pm10['Tinggi'], f_no2['Tinggi'], f_co['Tinggi'], f_so2['Tinggi'])))  # Aturan 2: Jika SEMUA polutan Tinggi → udara Buruk
    # R3
    rules.append(('Sedang', min(f_pm25['Sedang'], f_pm10['Sedang'], f_no2['Sedang'], f_co['Sedang'], f_so2['Sedang'])))  # Aturan 3: Jika SEMUA polutan Sedang → udara Sedang
    # R4
    rules.append(('Sedang', min(f_pm25['Tinggi'], f_pm10['Rendah'], f_no2['Rendah'], f_co['Rendah'], f_so2['Rendah'])))  # Aturan 4: PM2.5 Tinggi tapi polutan lain Rendah → masih Sedang (PM2.5 dominan tapi tidak semua buruk)
    # R5
    rules.append(('Sedang', min(f_pm25['Rendah'], f_pm10['Tinggi'], f_no2['Rendah'], f_co['Rendah'], f_so2['Rendah'])))  # Aturan 5: Hanya PM10 yang Tinggi → Sedang
    # R6
    rules.append(('Sedang', min(f_pm25['Rendah'], f_pm10['Rendah'], f_no2['Tinggi'], f_co['Rendah'], f_so2['Rendah'])))  # Aturan 6: Hanya NO2 yang Tinggi → Sedang
    # R7
    rules.append(('Buruk', min(f_pm25['Tinggi'], f_pm10['Tinggi'], f_no2['Rendah'], f_co['Rendah'], f_so2['Rendah'])))  # Aturan 7: PM2.5 dan PM10 sama-sama Tinggi → Buruk (dua partikel berbahaya sekaligus)
    # R8
    rules.append(('Buruk', min(f_pm25['Sedang'], f_pm10['Tinggi'], f_no2['Sedang'], f_co['Rendah'], f_so2['Rendah'])))  # Aturan 8: PM10 Tinggi + PM2.5 dan NO2 Sedang → Buruk
    # R9
    rules.append(('Sedang', min(f_pm25['Rendah'], f_pm10['Rendah'], f_no2['Rendah'], f_co['Tinggi'], f_so2['Tinggi'])))  # Aturan 9: CO dan SO2 sama-sama Tinggi tapi partikel rendah → Sedang
    # R10
    rules.append(('Buruk', min(f_pm25['Tinggi'], f_pm10['Sedang'], f_no2['Tinggi'], f_co['Sedang'], f_so2['Rendah'])))  # Aturan 10: PM2.5 dan NO2 Tinggi, PM10 dan CO Sedang → Buruk
    # R11
    rules.append(('Baik', min(f_pm25['Rendah'], f_pm10['Sedang'], f_no2['Rendah'], f_co['Rendah'], f_so2['Sedang'])))  # Aturan 11: Hanya PM10 dan SO2 yang Sedang, sisanya Rendah → masih Baik
    # R12
    rules.append(('Baik', min(f_pm25['Sedang'], f_pm10['Rendah'], f_no2['Sedang'], f_co['Rendah'], f_so2['Rendah'])))  # Aturan 12: PM2.5 dan NO2 Sedang tapi yang lain Rendah → masih Baik
    # R13
    rules.append(('Buruk', min(f_pm25['Tinggi'], f_pm10['Rendah'], f_no2['Tinggi'], f_co['Rendah'], f_so2['Tinggi'])))  # Aturan 13: PM2.5, NO2, dan SO2 semuanya Tinggi → Buruk
    # R14
    rules.append(('Sedang', min(f_pm25['Rendah'], f_pm10['Tinggi'], f_no2['Rendah'], f_co['Tinggi'], f_so2['Rendah'])))  # Aturan 14: PM10 dan CO sama-sama Tinggi → Sedang
    # R15
    rules.append(('Sedang', min(f_pm25['Sedang'], f_pm10['Sedang'], f_no2['Rendah'], f_co['Sedang'], f_so2['Rendah'])))  # Aturan 15: PM2.5, PM10, CO Sedang, yang lain Rendah → Sedang

    out_baik = max([r[1] for r in rules if r[0] == 'Baik'])  # Kumpulkan semua aturan yang hasilnya 'Baik', ambil nilai terbesarnya (metode MAX)
    out_sedang = max([r[1] for r in rules if r[0] == 'Sedang'])  # Kumpulkan semua aturan yang hasilnya 'Sedang', ambil nilai terbesarnya
    out_buruk = max([r[1] for r in rules if r[0] == 'Buruk'])  # Kumpulkan semua aturan yang hasilnya 'Buruk', ambil nilai terbesarnya
    
    return out_baik, out_sedang, out_buruk  # Kembalikan tiga nilai keanggotaan output untuk digunakan di defuzzifikasi

# 5. DEFUZZIFIKASI
def defuzzify_mamdani(out_baik, out_sedang, out_buruk):  # Fungsi untuk mengubah nilai keanggotaan fuzzy menjadi angka AQI nyata (metode Mamdani)
    numerator = 0  # Variabel untuk menghitung jumlah perkalian x dengan derajat keanggotaan (pembilang)
    denominator = 0  # Variabel untuk menghitung total derajat keanggotaan (penyebut)
    for x in range(0, 501, 5):  # Cek setiap titik dari 0 sampai 500, dengan langkah 5 (cukup detail tapi tidak terlalu berat)
        mu_baik = min(out_baik, trapmf(x, 0, 0, 50, 100))  # Batasi tinggi kurva 'Baik' berdasarkan nilai inferensi (proses clipping)
        mu_sedang = min(out_sedang, trapmf(x, 50, 100, 200, 300))  # Batasi tinggi kurva 'Sedang' berdasarkan nilai inferensi
        mu_buruk = min(out_buruk, trapmf(x, 200, 300, 500, 500))  # Batasi tinggi kurva 'Buruk' berdasarkan nilai inferensi
        
        mu_agg = max(mu_baik, mu_sedang, mu_buruk)  # Gabungkan tiga kurva dengan ambil nilai tertinggi di titik x (agregasi)
        
        numerator += x * mu_agg  # Tambahkan hasil perkalian posisi x dengan derajat keanggotaan ke pembilang
        denominator += mu_agg  # Tambahkan derajat keanggotaan ke penyebut
        
    if denominator == 0: return 0  # Jika tidak ada keanggotaan sama sekali (semua nol), kembalikan 0 untuk menghindari pembagian nol
    return numerator / denominator  # Hitung titik berat (centroid) = rata-rata tertimbang → ini adalah nilai AQI hasil prediksi

def defuzzify_sugeno(out_baik, out_sedang, out_buruk):  # Fungsi defuzzifikasi metode Sugeno (lebih simpel dari Mamdani)
    z_baik, z_sedang, z_buruk = 50, 150, 300  # Nilai tetap (konstanta) untuk mewakili masing-masing kategori: Baik=50, Sedang=150, Buruk=300
    numerator = (out_baik * z_baik) + (out_sedang * z_sedang) + (out_buruk * z_buruk)  # Kalikan setiap nilai keanggotaan dengan konstanta kategorinya lalu jumlahkan
    denominator = out_baik + out_sedang + out_buruk  # Jumlahkan semua nilai keanggotaan sebagai pembagi
    
    if denominator == 0: return 0  # Jika semua keanggotaan nol, kembalikan 0 agar tidak error
    return numerator / denominator  # Hitung rata-rata tertimbang → nilai AQI hasil prediksi Sugeno

# 6. EKSEKUSI & EVALUASI
pred_mamdani = []  # Siapkan daftar kosong untuk menampung hasil prediksi metode Mamdani
pred_sugeno = []  # Siapkan daftar kosong untuk menampung hasil prediksi metode Sugeno
actual_aqi = data_sample['AQI'].values  # Ambil nilai AQI asli dari data sebagai pembanding

for i, row in data_sample.iterrows():  # Proses setiap baris data satu per satu
    f_pm25 = fuzzify_input(row['PM2.5'], 'PM2.5')  # Ubah nilai PM2.5 menjadi derajat keanggotaan (Rendah/Sedang/Tinggi)
    f_pm10 = fuzzify_input(row['PM10'], 'PM10')  # Ubah nilai PM10 menjadi derajat keanggotaan
    f_no2 = fuzzify_input(row['NO2'], 'NO2')  # Ubah nilai NO2 menjadi derajat keanggotaan
    f_co = fuzzify_input(row['CO'], 'CO')  # Ubah nilai CO menjadi derajat keanggotaan
    f_so2 = fuzzify_input(row['SO2'], 'SO2')  # Ubah nilai SO2 menjadi derajat keanggotaan
    
    out_baik, out_sedang, out_buruk = apply_rules(f_pm25, f_pm10, f_no2, f_co, f_so2)  # Terapkan semua aturan fuzzy, hasilkan kekuatan masing-masing kategori output
    
    pred_mamdani.append(defuzzify_mamdani(out_baik, out_sedang, out_buruk))  # Hitung prediksi AQI pakai metode Mamdani, simpan ke daftar
    pred_sugeno.append(defuzzify_sugeno(out_baik, out_sedang, out_buruk))  # Hitung prediksi AQI pakai metode Sugeno, simpan ke daftar

mae_mamdani = mean_absolute_error(actual_aqi, pred_mamdani)  # Hitung MAE Mamdani: rata-rata selisih absolut antara prediksi dan nilai asli
mse_mamdani = mean_squared_error(actual_aqi, pred_mamdani)  # Hitung MSE Mamdani: rata-rata kuadrat selisih (lebih sensitif terhadap kesalahan besar)

mae_sugeno = mean_absolute_error(actual_aqi, pred_sugeno)  # Hitung MAE Sugeno
mse_sugeno = mean_squared_error(actual_aqi, pred_sugeno)  # Hitung MSE Sugeno

print(f"EVALUASI PERFORMA")  # Cetak judul evaluasi
print(f"Mamdani -> MAE: {mae_mamdani:.2f} | MSE: {mse_mamdani:.2f}")  # Tampilkan nilai MAE dan MSE Mamdani (2 angka desimal)
print(f"Sugeno  -> MAE: {mae_sugeno:.2f} | MSE: {mse_sugeno:.2f}")  # Tampilkan nilai MAE dan MSE Sugeno

print("\n" + "="*40)  # Cetak baris kosong + garis pemisah 40 karakter
print("TABEL PREDIKSI (10 SAMPEL PERTAMA)")  # Cetak judul tabel
print("="*40)  # Cetak garis pemisah bawah

df_hasil = data_sample[['PM2.5', 'PM10', 'NO2', 'CO', 'SO2']].copy()  # Buat tabel baru berisi data polutan saja (salin agar tidak mengubah data asli)
df_hasil['Actual AQI'] = actual_aqi  # Tambahkan kolom nilai AQI asli dari dataset
df_hasil['Prediksi Mamdani'] = pred_mamdani  # Tambahkan kolom hasil prediksi Mamdani
df_hasil['Prediksi Sugeno'] = pred_sugeno  # Tambahkan kolom hasil prediksi Sugeno

df_hasil['Prediksi Mamdani'] = df_hasil['Prediksi Mamdani'].round(2)  # Bulatkan hasil Mamdani jadi 2 desimal agar lebih rapi
df_hasil['Prediksi Sugeno'] = df_hasil['Prediksi Sugeno'].round(2)  # Bulatkan hasil Sugeno jadi 2 desimal
df_hasil['Selisih (Mam-Sug)'] = abs(df_hasil['Prediksi Mamdani'] - df_hasil['Prediksi Sugeno']).round(2)  # Hitung selisih antara prediksi Mamdani dan Sugeno (nilai absolut, tanpa tanda minus)

print(df_hasil.head(10).to_string())  # Tampilkan 10 baris pertama dalam format teks yang rapi

EVALUASI PERFORMA
Mamdani -> MAE: 92.10 | MSE: 19570.24
Sugeno  -> MAE: 75.60 | MSE: 16116.08

TABEL PREDIKSI (10 SAMPEL PERTAMA)
   PM2.5    PM10    NO2    CO    SO2  Actual AQI  Prediksi Mamdani  Prediksi Sugeno  Selisih (Mam-Sug)
0  20.71   41.14  30.13  0.85   5.78        69.0             40.17             50.0               9.83
1  24.32  141.78  20.32  0.40   4.82       120.0            165.74            150.0              15.74
2  89.14  172.04  19.79  0.65   5.92       195.0            371.47            300.0              71.47
3  18.94   79.08  31.69  1.02   4.68        64.0             46.18             50.0               3.82
4  65.82  183.77  61.45  1.30  20.59       175.0            361.74            300.0              61.74
5  37.57   97.74  35.15  0.54   6.78        95.0             50.00             50.0               0.00
6  36.68   72.67  18.64  0.68  10.68        60.0             44.74             50.0               5.26
7  69.33  127.59  40.86  0.91  16.11       122

*****Kode output sugeno sudah disesuaikan*****

In [2]:
import pandas as pd  # Impor pustaka pandas untuk membaca dan mengolah data tabel
import numpy as np  # Impor numpy untuk operasi matematika
from sklearn.metrics import mean_absolute_error, mean_squared_error  # Impor fungsi pengukur kesalahan prediksi

# 1. Load & Preprocessing Data
df = pd.read_csv('city_day.csv')  # Baca file CSV berisi data kualitas udara kota per hari
df_clean = df[['PM2.5', 'PM10', 'NO2', 'CO', 'SO2', 'AQI']].dropna()  # Pilih 5 kolom polutan + AQI, buang baris yang ada nilainya kosong
data_sample = df_clean.sample(n=5000, random_state=42).reset_index(drop=True)  # Ambil 5000 data secara acak (agar proses tidak terlalu lama), random_state=42 agar hasilnya sama setiap dijalankan

# 2. FUNGSI KEANGGOTAAN (From Scratch)
def trapmf(x, a, b, c, d):  # Fungsi bentuk trapesium: mengukur seberapa besar nilai x termasuk dalam suatu kategori
    if x <= a or x >= d: return 0.0  # Kalau x di luar batas kiri (a) atau kanan (d), berarti tidak masuk kategori sama sekali (nilai 0)
    elif a < x <= b: return (x - a) / (b - a)  # Sisi kiri naik: semakin mendekati b, semakin besar nilainya
    elif b < x <= c: return 1.0  # Di antara b dan c, nilai keanggotaan penuh (= 1, artinya benar-benar masuk kategori ini)
    elif c < x < d: return (d - x) / (d - c)  # Sisi kanan turun: semakin mendekati d, semakin kecil nilainya
    return 0.0  # Jaga-jaga jika tidak ada kondisi yang cocok, kembalikan 0

def trimf(x, a, b, c):  # Fungsi bentuk segitiga: sebenarnya trapesium tapi dengan puncak setitik (b=c)
    return trapmf(x, a, b, b, c)  # Gunakan trapmf tapi titik tengah kiri dan kanan sama (b), membentuk segitiga

# 3. PROSES FUZZIFIKASI
def fuzzify_input(val, param):  # Fungsi untuk mengubah nilai angka menjadi derajat keanggotaan (Rendah/Sedang/Tinggi)
    if param == 'PM2.5':  # Jika parameter yang diproses adalah polutan PM2.5
        return {'Rendah': trapmf(val, 0, 0, 30, 60), 'Sedang': trimf(val, 30, 60, 90), 'Tinggi': trapmf(val, 60, 90, 500, 500)}  # Kembalikan seberapa 'Rendah', 'Sedang', 'Tinggi' nilai PM2.5 tersebut
    elif param == 'PM10':  # Jika parameter yang diproses adalah polutan PM10
        return {'Rendah': trapmf(val, 0, 0, 50, 100), 'Sedang': trimf(val, 50, 100, 150), 'Tinggi': trapmf(val, 100, 150, 500, 500)}  # Batas kategori PM10 berbeda karena standar polutan berbeda
    elif param == 'NO2':  # Jika parameter yang diproses adalah polutan NO2
        return {'Rendah': trapmf(val, 0, 0, 40, 80), 'Sedang': trimf(val, 40, 80, 120), 'Tinggi': trapmf(val, 80, 120, 300, 300)}  # Batas kategori untuk gas NO2
    elif param == 'CO':  # Jika parameter yang diproses adalah polutan CO (karbon monoksida)
        return {'Rendah': trapmf(val, 0, 0, 1, 2), 'Sedang': trimf(val, 1, 2, 3), 'Tinggi': trapmf(val, 2, 3, 50, 50)}  # CO diukur dalam satuan kecil, makanya batasnya jauh lebih kecil
    elif param == 'SO2':  # Jika parameter yang diproses adalah polutan SO2
        return {'Rendah': trapmf(val, 0, 0, 40, 80), 'Sedang': trimf(val, 40, 80, 120), 'Tinggi': trapmf(val, 80, 120, 300, 300)}  # Batas kategori SO2 sama dengan NO2

# 4. INFERENSI & RULE BASE
def apply_rules(f_pm25, f_pm10, f_no2, f_co, f_so2):  # Fungsi untuk menerapkan 15 aturan fuzzy berdasarkan nilai keanggotaan semua polutan
    rules = []  # Buat daftar kosong untuk menampung hasil tiap aturan
    
    # R1
    rules.append(('Baik', min(f_pm25['Rendah'], f_pm10['Rendah'], f_no2['Rendah'], f_co['Rendah'], f_so2['Rendah'])))  # Aturan 1: Jika SEMUA polutan Rendah → udara Baik
    # R2
    rules.append(('Buruk', min(f_pm25['Tinggi'], f_pm10['Tinggi'], f_no2['Tinggi'], f_co['Tinggi'], f_so2['Tinggi'])))  # Aturan 2: Jika SEMUA polutan Tinggi → udara Buruk
    # R3
    rules.append(('Sedang', min(f_pm25['Sedang'], f_pm10['Sedang'], f_no2['Sedang'], f_co['Sedang'], f_so2['Sedang'])))  # Aturan 3: Jika SEMUA polutan Sedang → udara Sedang
    # R4
    rules.append(('Sedang', min(f_pm25['Tinggi'], f_pm10['Rendah'], f_no2['Rendah'], f_co['Rendah'], f_so2['Rendah'])))  # Aturan 4: PM2.5 Tinggi tapi polutan lain Rendah → masih Sedang
    # R5
    rules.append(('Sedang', min(f_pm25['Rendah'], f_pm10['Tinggi'], f_no2['Rendah'], f_co['Rendah'], f_so2['Rendah'])))  # Aturan 5: Hanya PM10 yang Tinggi → Sedang
    # R6
    rules.append(('Sedang', min(f_pm25['Rendah'], f_pm10['Rendah'], f_no2['Tinggi'], f_co['Rendah'], f_so2['Rendah'])))  # Aturan 6: Hanya NO2 yang Tinggi → Sedang
    # R7
    rules.append(('Buruk', min(f_pm25['Tinggi'], f_pm10['Tinggi'], f_no2['Rendah'], f_co['Rendah'], f_so2['Rendah'])))  # Aturan 7: PM2.5 dan PM10 sama-sama Tinggi → Buruk
    # R8
    rules.append(('Buruk', min(f_pm25['Sedang'], f_pm10['Tinggi'], f_no2['Sedang'], f_co['Rendah'], f_so2['Rendah'])))  # Aturan 8: PM10 Tinggi + PM2.5 dan NO2 Sedang → Buruk
    # R9
    rules.append(('Sedang', min(f_pm25['Rendah'], f_pm10['Rendah'], f_no2['Rendah'], f_co['Tinggi'], f_so2['Tinggi'])))  # Aturan 9: CO dan SO2 sama-sama Tinggi tapi partikel rendah → Sedang
    # R10
    rules.append(('Buruk', min(f_pm25['Tinggi'], f_pm10['Sedang'], f_no2['Tinggi'], f_co['Sedang'], f_so2['Rendah'])))  # Aturan 10: PM2.5 dan NO2 Tinggi, PM10 dan CO Sedang → Buruk
    # R11
    rules.append(('Baik', min(f_pm25['Rendah'], f_pm10['Sedang'], f_no2['Rendah'], f_co['Rendah'], f_so2['Sedang'])))  # Aturan 11: Hanya PM10 dan SO2 yang Sedang, sisanya Rendah → masih Baik
    # R12
    rules.append(('Baik', min(f_pm25['Sedang'], f_pm10['Rendah'], f_no2['Sedang'], f_co['Rendah'], f_so2['Rendah'])))  # Aturan 12: PM2.5 dan NO2 Sedang tapi yang lain Rendah → masih Baik
    # R13
    rules.append(('Buruk', min(f_pm25['Tinggi'], f_pm10['Rendah'], f_no2['Tinggi'], f_co['Rendah'], f_so2['Tinggi'])))  # Aturan 13: PM2.5, NO2, dan SO2 semuanya Tinggi → Buruk
    # R14
    rules.append(('Sedang', min(f_pm25['Rendah'], f_pm10['Tinggi'], f_no2['Rendah'], f_co['Tinggi'], f_so2['Rendah'])))  # Aturan 14: PM10 dan CO sama-sama Tinggi → Sedang
    # R15
    rules.append(('Sedang', min(f_pm25['Sedang'], f_pm10['Sedang'], f_no2['Rendah'], f_co['Sedang'], f_so2['Rendah'])))  # Aturan 15: PM2.5, PM10, CO Sedang, yang lain Rendah → Sedang

    out_baik = max([r[1] for r in rules if r[0] == 'Baik'])  # Kumpulkan semua aturan yang hasilnya 'Baik', ambil nilai terbesarnya (metode MAX)
    out_sedang = max([r[1] for r in rules if r[0] == 'Sedang'])  # Kumpulkan semua aturan yang hasilnya 'Sedang', ambil nilai terbesarnya
    out_buruk = max([r[1] for r in rules if r[0] == 'Buruk'])  # Kumpulkan semua aturan yang hasilnya 'Buruk', ambil nilai terbesarnya
    
    return out_baik, out_sedang, out_buruk  # Kembalikan tiga nilai keanggotaan output

# 5. DEFUZZIFIKASI
def defuzzify_mamdani(out_baik, out_sedang, out_buruk):  # Fungsi defuzzifikasi Mamdani dengan metode Centroid (titik berat)
    numerator = 0  # Variabel pembilang untuk rumus centroid
    denominator = 0  # Variabel penyebut untuk rumus centroid
    for x in range(0, 501, 5):  # Cek setiap titik dari 0 sampai 500 dengan langkah 5
        # Kurva output dibuat asimetris agar titik berat mendekati angka target (50, 150, 300)
        mu_baik = min(out_baik, trapmf(x, 0, 0, 60, 120))         # Centroid aslinya ~ 46.6 — batasi tinggi kurva 'Baik' sesuai nilai inferensi
        mu_sedang = min(out_sedang, trapmf(x, 40, 100, 160, 280)) # Centroid aslinya ~ 148.0 — batasi tinggi kurva 'Sedang' sesuai nilai inferensi
        mu_buruk = min(out_buruk, trapmf(x, 50, 150, 500, 500))   # Centroid aslinya ~ 299.0 — batasi tinggi kurva 'Buruk' sesuai nilai inferensi
        
        mu_agg = max(mu_baik, mu_sedang, mu_buruk)  # Gabungkan tiga kurva dengan ambil nilai tertinggi di titik x (agregasi)
        
        numerator += x * mu_agg  # Tambahkan hasil perkalian posisi x dengan derajat keanggotaan ke pembilang
        denominator += mu_agg  # Tambahkan derajat keanggotaan ke penyebut
        
    if denominator == 0: return 0  # Jika tidak ada keanggotaan sama sekali, kembalikan 0 untuk menghindari error pembagian nol
    return numerator / denominator  # Hitung titik berat (centroid) → ini adalah nilai AQI hasil prediksi Mamdani

def defuzzify_sugeno(out_baik, out_sedang, out_buruk):  # Fungsi defuzzifikasi Sugeno dengan rata-rata tertimbang (lebih cepat dari Mamdani)
    z_baik, z_sedang, z_buruk = 50, 150, 300  # Nilai tetap (konstanta) mewakili tiap kategori: Baik=50, Sedang=150, Buruk=300
    numerator = (out_baik * z_baik) + (out_sedang * z_sedang) + (out_buruk * z_buruk)  # Kalikan keanggotaan × konstanta tiap kategori lalu jumlahkan
    denominator = out_baik + out_sedang + out_buruk  # Jumlahkan semua nilai keanggotaan sebagai pembagi
    
    if denominator == 0: return 0  # Jika semua keanggotaan nol, kembalikan 0 agar tidak error
    return numerator / denominator  # Hitung rata-rata tertimbang → nilai AQI hasil prediksi Sugeno

# 6. EKSEKUSI & EVALUASI
pred_mamdani = []  # Siapkan daftar kosong untuk menampung hasil prediksi metode Mamdani
pred_sugeno = []  # Siapkan daftar kosong untuk menampung hasil prediksi metode Sugeno
actual_aqi = data_sample['AQI'].values  # Ambil nilai AQI asli dari data sebagai pembanding

for i, row in data_sample.iterrows():  # Proses setiap baris data satu per satu
    f_pm25 = fuzzify_input(row['PM2.5'], 'PM2.5')  # Ubah nilai PM2.5 menjadi derajat keanggotaan
    f_pm10 = fuzzify_input(row['PM10'], 'PM10')  # Ubah nilai PM10 menjadi derajat keanggotaan
    f_no2 = fuzzify_input(row['NO2'], 'NO2')  # Ubah nilai NO2 menjadi derajat keanggotaan
    f_co = fuzzify_input(row['CO'], 'CO')  # Ubah nilai CO menjadi derajat keanggotaan
    f_so2 = fuzzify_input(row['SO2'], 'SO2')  # Ubah nilai SO2 menjadi derajat keanggotaan
    
    out_baik, out_sedang, out_buruk = apply_rules(f_pm25, f_pm10, f_no2, f_co, f_so2)  # Terapkan semua aturan fuzzy, hasilkan kekuatan tiap kategori output
    
    pred_mamdani.append(defuzzify_mamdani(out_baik, out_sedang, out_buruk))  # Hitung prediksi AQI metode Mamdani, simpan ke daftar
    pred_sugeno.append(defuzzify_sugeno(out_baik, out_sedang, out_buruk))  # Hitung prediksi AQI metode Sugeno, simpan ke daftar

mae_mamdani = mean_absolute_error(actual_aqi, pred_mamdani)  # Hitung MAE Mamdani: rata-rata selisih absolut antara prediksi dan nilai asli
mse_mamdani = mean_squared_error(actual_aqi, pred_mamdani)  # Hitung MSE Mamdani: rata-rata kuadrat selisih

mae_sugeno = mean_absolute_error(actual_aqi, pred_sugeno)  # Hitung MAE Sugeno
mse_sugeno = mean_squared_error(actual_aqi, pred_sugeno)  # Hitung MSE Sugeno

print(f"EVALUASI PERFORMA")  # Cetak judul evaluasi
print(f"Mamdani -> MAE: {mae_mamdani:.2f} | MSE: {mse_mamdani:.2f}")  # Tampilkan MAE dan MSE Mamdani
print(f"Sugeno  -> MAE: {mae_sugeno:.2f} | MSE: {mse_sugeno:.2f}")  # Tampilkan MAE dan MSE Sugeno

print("\n" + "="*40)  # Cetak baris kosong + garis pemisah
print("TABEL PREDIKSI (10 SAMPEL PERTAMA)")  # Cetak judul tabel
print("="*40)  # Cetak garis pemisah bawah

df_hasil = data_sample[['PM2.5', 'PM10', 'NO2', 'CO', 'SO2']].copy()  # Buat tabel baru berisi data polutan (salin agar tidak mengubah data asli)
df_hasil['Actual AQI'] = actual_aqi  # Tambahkan kolom nilai AQI asli
df_hasil['Prediksi Mamdani'] = pred_mamdani  # Tambahkan kolom hasil prediksi Mamdani
df_hasil['Prediksi Sugeno'] = pred_sugeno  # Tambahkan kolom hasil prediksi Sugeno

df_hasil['Prediksi Mamdani'] = df_hasil['Prediksi Mamdani'].round(2)  # Bulatkan hasil Mamdani jadi 2 desimal
df_hasil['Prediksi Sugeno'] = df_hasil['Prediksi Sugeno'].round(2)  # Bulatkan hasil Sugeno jadi 2 desimal
df_hasil['Selisih (Mam-Sug)'] = abs(df_hasil['Prediksi Mamdani'] - df_hasil['Prediksi Sugeno']).round(2)  # Hitung selisih absolut antara prediksi Mamdani dan Sugeno

print(df_hasil.head(10).to_string())  # Tampilkan 10 baris pertama dalam format teks yang rapi

EVALUASI PERFORMA
Mamdani -> MAE: 75.14 | MSE: 15946.70
Sugeno  -> MAE: 75.60 | MSE: 16116.08

TABEL PREDIKSI (10 SAMPEL PERTAMA)
   PM2.5    PM10    NO2    CO    SO2  Actual AQI  Prediksi Mamdani  Prediksi Sugeno  Selisih (Mam-Sug)
0  20.71   41.14  30.13  0.85   5.78        69.0             47.95             50.0               2.05
1  24.32  141.78  20.32  0.40   4.82       120.0            149.37            150.0               0.63
2  89.14  172.04  19.79  0.65   5.92       195.0            297.06            300.0               2.94
3  18.94   79.08  31.69  1.02   4.68        64.0             55.21             50.0               5.21
4  65.82  183.77  61.45  1.30  20.59       175.0            286.89            300.0              13.11
5  37.57   97.74  35.15  0.54   6.78        95.0             60.00             50.0              10.00
6  36.68   72.67  18.64  0.68  10.68        60.0             53.45             50.0               3.45
7  69.33  127.59  40.86  0.91  16.11       122